In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from math import radians, sin, cos, sqrt, atan2

# -------------------------------
# 1. Load Dataset
# -------------------------------
df = pd.read_csv("creditcard.csv")

# -------------------------------
# 2. Basic Features
# -------------------------------
df['hour'] = (df['Time'] // 3600) % 24

# -------------------------------
# 3. Assign 20 Users
# -------------------------------
df['user_id'] = np.random.randint(1, 21, size=len(df))

# -------------------------------
# 4. Base Locations (City Centers)
# -------------------------------
locations = {
    1: (18.5204, 73.8567), 2: (19.9975, 73.7898), 3: (19.0760, 72.8777),
    4: (28.7041, 77.1025), 5: (12.9716, 77.5946), 6: (13.0827, 80.2707),
    7: (17.3850, 78.4867), 8: (22.5726, 88.3639), 9: (23.0225, 72.5714),
    10: (26.9124, 75.7873), 11: (21.1702, 72.8311), 12: (21.1458, 79.0882),
    13: (22.7196, 75.8577), 14: (23.2599, 77.4126), 15: (26.8467, 80.9462),
    16: (26.4499, 80.3319), 17: (25.5941, 85.1376), 18: (30.7333, 76.7794),
    19: (11.0168, 76.9558), 20: (9.9312, 76.2673)
}

# -------------------------------
# 5. Random Location per Transaction
# -------------------------------
RANGE = 0.05

lat_list, lon_list = [], []

for _, row in df.iterrows():
    base_lat, base_lon = locations[row['user_id']]
    
    lat = base_lat + np.random.uniform(-RANGE, RANGE)
    lon = base_lon + np.random.uniform(-RANGE, RANGE)
    
    lat_list.append(lat)
    lon_list.append(lon)

df['latitude'] = lat_list
df['longitude'] = lon_list

# -------------------------------
# 6. Sort for time-based features
# -------------------------------
df = df.sort_values(by=['user_id', 'Time'])

# -------------------------------
# 7. txn_count_1hr
# -------------------------------
df['txn_count_1hr'] = 0

for user in df['user_id'].unique():
    user_df = df[df['user_id'] == user]
    times = user_df['Time'].values
    
    counts = []
    for i in range(len(times)):
        current_time = times[i]
        count = np.sum((times >= current_time - 3600) & (times <= current_time))
        counts.append(count)
    
    df.loc[user_df.index, 'txn_count_1hr'] = counts

# -------------------------------
# 8. Rolling Features
# -------------------------------
df['rolling_avg'] = df.groupby('user_id')['Amount'] \
    .transform(lambda x: x.rolling(5, min_periods=1).mean())

df['amount_deviation'] = df['Amount'] - df['rolling_avg']

# -------------------------------
# 9. Distance Calculation (IMPORTANT)
# -------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c

df['prev_lat'] = df.groupby('user_id')['latitude'].shift(1)
df['prev_lon'] = df.groupby('user_id')['longitude'].shift(1)

df['distance_km'] = df.apply(
    lambda row: haversine(
        row['latitude'], row['longitude'],
        row['prev_lat'], row['prev_lon']
    ) if pd.notnull(row['prev_lat']) else 0,
    axis=1
)

# -------------------------------
# 10. Velocity Feature
# -------------------------------
df['time_diff'] = df.groupby('user_id')['Time'].diff()

# Replace 0 or negative with small value
df['time_diff'] = df['time_diff'].replace(0, 1)
df['time_diff'] = df['time_diff'].fillna(1)

# Replace infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with NaN
df.dropna(inplace=True)

# Calculate velocity safely
df['velocity'] = df['distance_km'] / df['time_diff']

# -------------------------------
# 11. High Amount Flag
# -------------------------------
df['high_amount'] = (df['Amount'] > df['Amount'].quantile(0.95)).astype(int)

# -------------------------------
# 12. Scaling
# -------------------------------
scaler = StandardScaler()
df[['Amount', 'Time']] = scaler.fit_transform(df[['Amount', 'Time']])

# -------------------------------
# 13. Balance Dataset
# -------------------------------
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

legit_sample = legit.sample(n=len(fraud) * 3, random_state=42)

df_balanced = pd.concat([fraud, legit_sample])
df_balanced = df_balanced.sample(frac=1, random_state=42)

# -------------------------------
# 14. Save
# -------------------------------
df_balanced.to_csv("processed.csv", index=False)

print("Advanced preprocessing completed!")

✅ Advanced preprocessing completed!


In [ ]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

# -------------------------------
# 1. Load Data
# -------------------------------
df = pd.read_csv("processed.csv")

print("Dataset Shape:", df.shape)

# -------------------------------
# 2. Features
# -------------------------------
features = [
    'Amount', 'Time', 'hour',
    'txn_count_1hr',
    'latitude', 'longitude',
    'rolling_avg', 'amount_deviation',
    'distance_km', 'velocity',
    'high_amount'
]

X = df[features]
y = df['Class']

# -------------------------------
# 3. Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
# 4. Handle Imbalance
# -------------------------------
fraud = sum(y_train == 1)
legit = sum(y_train == 0)

scale_pos_weight = (legit / fraud) * 5

print("Scale Pos Weight:", scale_pos_weight)

# -------------------------------
# 5. Model (STRONG)
# -------------------------------
model = XGBClassifier(
    n_estimators=900,
    max_depth=12,
    learning_rate=0.015,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)

# -------------------------------
# 6. Train
# -------------------------------
model.fit(X_train, y_train)

# -------------------------------
# 7. Predict with THRESHOLD
# -------------------------------
y_prob = model.predict_proba(X_test)[:, 1]

# KEY CHANGE
threshold = 0.15
y_pred = (y_prob > threshold).astype(int)

# -------------------------------
# 8. Evaluation
# -------------------------------
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))

# -------------------------------
# 9. Save Model
# -------------------------------
pickle.dump(model, open("fraud_model.pkl", "wb"))

print("\nModel saved successfully!")

Dataset Shape: (1968, 44)
Scale Pos Weight: 14.974619289340101

Confusion Matrix:
[[203  93]
 [ 23  75]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.69      0.78       296
           1       0.45      0.77      0.56        98

    accuracy                           0.71       394
   macro avg       0.67      0.73      0.67       394
weighted avg       0.79      0.71      0.72       394


ROC-AUC Score:
0.8097766133480419

✅ Model saved successfully!
